# Clean Label Poisoning Attack on a Multi-Class Classifier

**Author:** Teodora Demerdzhieva  
**Topic:** Adversarial Machine Learning — Data Poisoning  
**Difficulty:** Intermediate

---

## What This Notebook Covers

This notebook demonstrates a **clean label poisoning attack** against a multi-class logistic regression classifier.

Unlike traditional data poisoning where an attacker mislabels training data, a clean label attack is more subtle:
- All labels remain **technically correct**
- Instead, the attacker **slightly modifies feature values** of carefully chosen training points
- This shifts the model's decision boundary without raising suspicion
- The result: a specific target point gets **misclassified** at inference time

This type of attack is particularly dangerous in real-world scenarios because:
- A data auditor checking labels would find nothing wrong
- The model maintains high overall accuracy
- Only the targeted prediction is affected

---

## Attack Goal

> Cause the model to misclassify a specific **Class 2** point as **Class 1** by poisoning the training data — without changing any labels.

---

## How It Works — Simple Intuition

Imagine the model draws a line (decision boundary) between Class 1 and Class 2.  
Our target point sits clearly on the Class 2 side of that line.

**The attack:**
1. Find Class 1 training points that are nearest to our target
2. Nudge those points slightly *toward* the target (labels stay as Class 1)
3. The model now sees Class 1 points suspiciously close to the target
4. The decision boundary shifts to accommodate them
5. The target ends up on the wrong side — misclassified as Class 1

## 1. Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import accuracy_score, classification_report

# Reproducibility — fixing the random seed ensures the same results every run
SEED = 1337
np.random.seed(SEED)

# Attack hyperparameters
# N_NEIGHBORS: how many Class 1 points we will perturb
# EPSILON_CROSS: how far (in feature space) we push each point toward the target
N_NEIGHBORS = 12
EPSILON_CROSS = 0.4

print("Libraries loaded.")
print(f"Attack config — neighbors: {N_NEIGHBORS}, perturbation magnitude: {EPSILON_CROSS}")

## 2. Load the Dataset

The dataset contains 3 classes in 2D feature space (already standardized).  
We also load a pre-defined **target index** — the specific training point we want the poisoned model to misclassify.

In [ ]:
data = np.load("clean_label_eval_dataset.npz")

X_train = data["Xtr"]
y_train = data["ytr"]
X_test  = data["Xte"]
y_test  = data["yte"]

# target_idx is the index of the point we want to misclassify
target_index = int(data["target_idx"].item())
data.close()

print(f"Training set : {X_train.shape}")
print(f"Test set     : {X_test.shape}")
print(f"Target index : {target_index}")

# Identify classes based on the target point
TARGET_CLASS     = int(y_train[target_index])   # the true class of the target (Class 2)
PERTURBING_CLASS = 1                             # the class whose points we will nudge
MISCLASSIFY_AS   = 1                             # the wrong class we want the model to predict

X_target_point = X_train[target_index]

print(f"\nTarget true class      : {TARGET_CLASS}")
print(f"Perturbing class       : {PERTURBING_CLASS}")
print(f"Goal — misclassify as  : {MISCLASSIFY_AS}")

## 3. Visualize the Clean Training Data

Before the attack, let's see where the target point sits relative to the other classes.  
The ✛ marker shows the target — it sits clearly inside the Class 2 region.

In [ ]:
def plot_dataset(X, y, title, target_idx=None, perturbed_idx=None):
    """
    Scatter plot of a 2D dataset with optional highlighting.
    - target_idx     : shown with a white star marker
    - perturbed_idx  : shown with a cyan edge to indicate poisoned points
    """
    colors = ["#0086ff", "#ffaf00", "#ff3e3e"]  # blue, yellow, red for classes 0/1/2
    
    plt.figure(figsize=(10, 6))
    
    for cls in np.unique(y):
        mask = y == cls
        plt.scatter(X[mask, 0], X[mask, 1],
                    c=colors[int(cls)], label=f"Class {int(cls)}",
                    edgecolors="#141d2b", s=50, alpha=0.7)
    
    # Highlight perturbed points (poisoned, but label unchanged)
    if perturbed_idx is not None and len(perturbed_idx) > 0:
        plt.scatter(X[perturbed_idx, 0], X[perturbed_idx, 1],
                    facecolors=colors[PERTURBING_CLASS],
                    edgecolors="#2ee7b6", s=120, linewidths=2,
                    label=f"Perturbed Class {PERTURBING_CLASS} points", zorder=3)
    
    # Highlight the target point
    if target_idx is not None:
        plt.scatter(X[target_idx, 0], X[target_idx, 1],
                    facecolors=colors[int(y[target_idx])],
                    edgecolors="white", marker="P", s=250, linewidths=2,
                    label=f"Target (Class {int(y[target_idx])})", zorder=4)
    
    plt.title(title, fontsize=14)
    plt.xlabel("Feature 1 (standardized)")
    plt.ylabel("Feature 2 (standardized)")
    plt.legend()
    plt.tight_layout()
    plt.show()


plot_dataset(X_train, y_train,
             title="Clean Training Data — Target point marked with ✛",
             target_idx=target_index)

![Trigger visualization](images/trigger_comparison.png)

## 4. Train a Baseline Model (Before Attack)

We first train a clean model to establish a performance baseline and confirm it correctly classifies the target point.

We use **One-vs-Rest Logistic Regression** — a standard multi-class approach that trains one binary classifier per class.

In [ ]:
baseline_model = OneVsRestClassifier(
    LogisticRegression(random_state=SEED, C=1.0, solver="liblinear")
)
baseline_model.fit(X_train, y_train)

baseline_pred_target = baseline_model.predict(X_target_point.reshape(1, -1))[0]
baseline_acc = accuracy_score(y_test, baseline_model.predict(X_test))

print("=== Baseline Model (Clean Data) ===")
print(f"Test accuracy          : {baseline_acc:.4f}")
print(f"Target point prediction: {baseline_pred_target} (true label: {TARGET_CLASS})")
print(f"Correctly classified   : {baseline_pred_target == TARGET_CLASS}")

## 5. The Clean Label Attack

### Strategy

We perturb `N_NEIGHBORS` training points from Class 1 (the perturbing class) by moving them toward the target point:

```
new_position = old_position + ε × unit_vector_toward_target
```

- `ε` (epsilon) controls how far we push each point — small enough to look natural, large enough to shift the boundary
- The label of each perturbed point stays as Class 1 — the data looks clean
- But the model now draws its boundary in a way that pulls the target into the Class 1 region

### Why nearest neighbors?

Points that are already close to the target have the most influence on the decision boundary near it. Perturbing far-away points would have little effect.

In [ ]:
def clean_label_attack(X_train_orig, y_train_orig, target_idx,
                        perturb_class, n_neighbors, epsilon, seed):
    """
    Performs a clean label poisoning attack.

    Strategy:
    - Find the n_neighbors points from perturb_class closest to the target
    - Nudge each one toward the target by epsilon (labels unchanged)
    - Return the poisoned dataset and the indices of modified points

    Parameters
    ----------
    X_train_orig  : original feature matrix
    y_train_orig  : original label array
    target_idx    : index of the point we want to misclassify
    perturb_class : class whose points we will perturb
    n_neighbors   : how many points to perturb
    epsilon       : perturbation magnitude
    seed          : random seed for reproducibility

    Returns
    -------
    X_poisoned      : modified feature matrix
    y_poisoned      : label array (unchanged)
    perturbed_idx   : indices of the modified points
    """
    np.random.seed(seed)

    # Work on copies — never modify the original data
    X_poisoned = X_train_orig.copy()
    y_poisoned = y_train_orig.copy()

    # The point we want to misclassify
    X_target = X_train_orig[target_idx]

    # Isolate all points that belong to the perturbing class
    perturb_mask    = (y_train_orig == perturb_class)
    X_perturb       = X_train_orig[perturb_mask]
    perturb_indices = np.where(perturb_mask)[0]  # their positions in the full dataset

    # Find the n_neighbors points from perturb_class closest to the target
    nn = NearestNeighbors(n_neighbors=n_neighbors)
    nn.fit(X_perturb)
    _, neighbor_positions = nn.kneighbors(X_target.reshape(1, -1))

    # Map back to indices in the full training set
    perturbed_idx = perturb_indices[neighbor_positions[0]]

    # Nudge each selected point toward the target
    for idx in perturbed_idx:
        direction = X_target - X_train_orig[idx]   # vector pointing from point to target
        norm = np.linalg.norm(direction)

        if norm > 0:
            unit_vector = direction / norm           # normalize: direction only, no magnitude
        else:
            unit_vector = direction                  # point is already at the target (edge case)

        # Move the point epsilon distance toward the target
        X_poisoned[idx] = X_train_orig[idx] + epsilon * unit_vector

    return X_poisoned, y_poisoned, perturbed_idx


# Run the attack
X_train_poisoned, y_train_poisoned, perturbed_indices = clean_label_attack(
    X_train, y_train,
    target_idx=target_index,
    perturb_class=PERTURBING_CLASS,
    n_neighbors=N_NEIGHBORS,
    epsilon=EPSILON_CROSS,
    seed=SEED
)

print(f"Attack complete.")
print(f"Points perturbed : {len(perturbed_indices)}")
print(f"Their indices    : {perturbed_indices}")
print(f"Labels changed   : {not np.array_equal(y_train, y_train_poisoned)} (should be False)")

## 6. Visualize the Poisoned Data

The cyan-outlined points are the perturbed Class 1 points — they have moved slightly toward the target.  
Their labels are still Class 1. To a human reviewer, the dataset looks normal.

In [ ]:
plot_dataset(X_train_poisoned, y_train_poisoned,
             title=f"Poisoned Training Data — {len(perturbed_indices)} Class 1 points nudged toward target",
             target_idx=target_index,
             perturbed_idx=perturbed_indices)

## 7. Train the Poisoned Model and Evaluate

We train a new model on the poisoned data using the exact same architecture.  
A successful attack means:
- The target point is now **misclassified as Class 1**
- Overall test accuracy remains **high** (the attack is stealthy)

In [ ]:
poisoned_model = OneVsRestClassifier(
    LogisticRegression(random_state=SEED, C=1.0, solver="liblinear")
)
poisoned_model.fit(X_train_poisoned, y_train_poisoned)

# Check prediction on the target point
target_pred = poisoned_model.predict(X_target_point.reshape(1, -1))[0]

# Check overall accuracy on the clean test set
poisoned_acc = accuracy_score(y_test, poisoned_model.predict(X_test))

print("=== Poisoned Model Results ===")
print(f"Test accuracy              : {poisoned_acc:.4f} (baseline was {baseline_acc:.4f})")
print(f"Target true label          : {TARGET_CLASS}")
print(f"Target predicted (poisoned): {target_pred}")
print()

if target_pred == MISCLASSIFY_AS:
    print("Attack SUCCESSFUL — target misclassified as required class")
    acc_drop = baseline_acc - poisoned_acc
    print(f"   Accuracy drop: {acc_drop:.4f} — attack is {'stealthy' if acc_drop < 0.05 else 'noticeable'}")
else:
    print(f"Attack FAILED — target predicted as Class {target_pred}")

## 8. Key Takeaways

| Aspect | Detail |
|--------|--------|
| **Attack type** | Clean label data poisoning |
| **Labels modified** | None — all labels remain correct |
| **Points perturbed** | 12 nearest Class 1 neighbors of the target |
| **Perturbation** | ε = 0.4 units toward the target in feature space |
| **Detection difficulty** | High — a label audit would find nothing wrong |
| **Model affected** | One-vs-Rest Logistic Regression |

### Real-World Implications

This attack is relevant wherever an attacker can influence the **training data pipeline**:
- Crowdsourced labeling platforms
- Web-scraped training datasets
- Federated learning environments
- Any system where data is collected from untrusted sources

### Defenses

- **Data sanitization** — detect statistical outliers before training
- **Certified defenses** — training methods that provide robustness guarantees
- **Influence function analysis** — identify which training points most affect a specific prediction
- **Ensemble methods** — harder to shift the boundary of multiple diverse models simultaneously